In [ ]:
import warnings
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import xgboost as xgb
import lightgbm as lgb
import catboost as ct
from sklearn.metrics import log_loss, confusion_matrix, roc_curve, roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split, GridSearchCV
from sklearn.impute import KNNImputer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import StackingClassifier, VotingClassifier
from catboost import CatBoostClassifier
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')

# Set up Kaggle dataset directory
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
def balanced_log_loss(y_true, y_pred):
    N_0 = np.sum(1 - y_true)
    N_1 = np.sum(y_true)
    p_1 = np.clip(y_pred, 1e-15, 1 - 1e-15)
    p_0 = 1 - p_1
    log_loss_0 = -np.sum((1 - y_true) * np.log(p_0))
    log_loss_1 = -np.sum(y_true * np.log(p_1))
    w_0 = 1 / N_0
    w_1 = 1 / N_1
    balanced_log_loss = 2*(w_0 * log_loss_0 + w_1 * log_loss_1) / (w_0 + w_1)
    return balanced_log_loss/(N_0+N_1)

## **Importing data**

In [ ]:
train = pd.read_csv('/kaggle/input/icr-identify-age-related-conditions/train.csv')
test = pd.read_csv('/kaggle/input/icr-identify-age-related-conditions/test.csv')
greeks = pd.read_csv('/kaggle/input/icr-identify-age-related-conditions/greeks.csv')
sample_submission = pd.read_csv('/kaggle/input/icr-identify-age-related-conditions/sample_submission.csv')

In [ ]:
train.head()

In [ ]:
greeks.head()

In [ ]:
train.loc[:, train.isna().mean()>0] .isna().mean()

In [ ]:
df = pd.merge(train, greeks, how='left', on='Id')

In [ ]:
df.head()

In [ ]:
df.describe().T

## **Data Cleaning**

In [ ]:
df['Class'].value_counts()

In [ ]:
df['Alpha'].value_counts()

In [ ]:
def encode(dataframe):
    le = LabelEncoder()
    obj = list(dataframe.loc[:, dataframe.dtypes == 'object'].columns)
    for i in obj:
        if i not in ['Id', 'Epsilon']:
            dataframe[i] = le.fit_transform(dataframe[i])
    return dataframe

In [ ]:
df = encode(df)  
test = encode(test)

In [ ]:
df.columns

In [ ]:
features = ['AB', 'AF', 'AH', 'AM', 'AR', 'AX', 'AY', 'AZ', 'BC', 'BD ', 'BN',
       'BP', 'BQ', 'BR', 'BZ', 'CB', 'CC', 'CD ', 'CF', 'CH', 'CL', 'CR', 'CS',
       'CU', 'CW ', 'DA', 'DE', 'DF', 'DH', 'DI', 'DL', 'DN', 'DU', 'DV', 'DY',
       'EB', 'EE', 'EG', 'EH', 'EJ', 'EL', 'EP', 'EU', 'FC', 'FD ', 'FE', 'FI',
       'FL', 'FR', 'FS', 'GB', 'GE', 'GF', 'GH', 'GI', 'GL']

target = 'Class'

In [ ]:
imputer = KNNImputer(n_neighbors=2)

df[features] = imputer.fit_transform(df[features])
test[features] = imputer.fit_transform(test[features])

In [ ]:
df.head()

## **Vizualization**

In [ ]:
correlation_matrix = df.corr().abs()

correlation_matrix = correlation_matrix[correlation_matrix < 1.0]

top_correlated_features = correlation_matrix.unstack().sort_values(ascending=False)[:100]

print(top_correlated_features)

In [ ]:
fig = plt.figure(figsize=(6*6, 45), dpi=130)
for idx, col in enumerate(features):
    ax = plt.subplot(19, 3, idx + 1)
    sns.kdeplot(
        data=df, hue='Class', fill=True,
        x=col,legend=False
    )
            
    ax.set_ylabel(''); ax.spines['top'].set_visible(False), 
    ax.set_xlabel(''); ax.spines['right'].set_visible(False)
    ax.set_title(f'{col}', loc='right', 
                 weight='bold', fontsize=20)

fig.suptitle(f'Features vs Class\n\n\n', ha='center',  fontweight='bold', fontsize=25)
fig.legend([1, 0], loc='upper center', bbox_to_anchor=(0.5, 0.97), fontsize=25, ncol=3)
plt.tight_layout()
plt.show()

## **Feature Engineering**

In [ ]:
#soon

## **Modeling**

In [ ]:
df.columns

In [ ]:
features = [ 'AB', 'AF', 'AH', 'AM', 'AR', 'AX', 'AY', 'AZ', 'BC', 'BD ', 'BN',
       'BP', 'BQ', 'BR', 'BZ', 'CB', 'CC', 'CD ', 'CF', 'CH', 'CL', 'CR', 'CS',
       'CU', 'CW ', 'DA', 'DE', 'DF', 'DH', 'DI', 'DL', 'DN', 'DU', 'DV', 'DY',
       'EB', 'EE', 'EG', 'EH', 'EJ', 'EL', 'EP', 'EU', 'FC', 'FD ', 'FE', 'FI',
       'FL', 'FR', 'FS', 'GB', 'GE', 'GF', 'GH', 'GI', 'GL'
        ]

target = 'Class'

In [ ]:
%%capture

xgb_params = {
    'colsample_bytree': 0.5, 
    'gamma': 1.0,
    'learning_rate': 0.01777187034634523,
    'max_depth': 6,
    'min_child_weight': 1,
    'n_estimators': 1500, 
    'subsample': 0.7629766636827013,
    'verbosity': 0,
    'random_state': 42,
    'tree_method': 'gpu_hist',
    'predictor': 'gpu_predictor'
}

xgb_params1 = {
    'colsample_bytree': 0.5,
    'gamma': 0.08898545568136436,
    'learning_rate': 0.009253274006068297,
    'max_depth': 3,
    'min_child_weight': 1,
    'n_estimators': 1500,
    'subsample': 0.8971494956585011,
    'verbosity': 0,
    'random_state': 42,
    'tree_method': 'gpu_hist',
    'predictor': 'gpu_predictor'
}

lgb_params = {
    'colsample_bytree': 0.5, 
    'learning_rate': 0.02, 
    'max_depth': 8, 
    'min_child_samples': 57,
    'n_estimators': 2445, 
    'num_leaves': 57,
    'reg_alpha': 0.6197994214239195, 
    'reg_lambda': 0.8675671389814725, 
    'subsample': 0.5264255077986388,
    'device': 'gpu',
    'random_state': 42,
    'verbose': -1 
}

lgb1_params = {
    'colsample_bytree': 0.5, 
    'learning_rate': 0.02,
    'max_depth': 4,
    'min_child_samples': 5,
    'n_estimators': 3000,
    'num_leaves': 100,
    'reg_alpha': 1.0,
    'reg_lambda': 1.0,
    'subsample': 0.5,
    'device': 'gpu',
    'random_state': 42,
    'verbose': -1 
} 

lgb2_params = {
    'colsample_bytree': 0.5,
    'learning_rate': 0.02,
    'max_depth': 4,
    'min_child_samples': 5,
    'n_estimators': 1476,
    'num_leaves': 100,
    'reg_alpha':  0.6362952390423132,
    'reg_lambda': 1.0, 
    'subsample': 1.0,
    'device': 'gpu',
    'random_state': 42,
    'verbose': -1 
} 


models = [
    ('xgb', xgb.XGBClassifier(**xgb_params)),
    ('xgb1', xgb.XGBClassifier(**xgb_params1)),
    ('lgb', lgb.LGBMClassifier(**lgb_params)),
    ('lgb1', lgb.LGBMClassifier(**lgb1_params)),
    ('lgb2', lgb.LGBMClassifier(**lgb2_params))
]

stacking_model = StackingClassifier(
        estimators=models[1:],
        final_estimator=xgb.XGBClassifier(**xgb_params),
        cv=5,
        stack_method='predict_proba',
        n_jobs=-1
)

voting_model = VotingClassifier(models, voting='soft')

models_iter = {
    'stacking': stacking_model,
    'voting': voting_model
}

# Load your dataset
X = df[features]
y = df[target]

skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
scores = []
m = []

for train_idx, val_idx in skf.split(X, y):
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_valid, y_valid = X.iloc[val_idx], y.iloc[val_idx]

    for model in models_iter.values():
        
        pipeline = Pipeline([
            ('scaler', MinMaxScaler()),
            ('model', model)
        ])
        pipeline.fit(X_train, y_train)
        val_preds = pipeline.predict_proba(X_valid)
        val_score = balanced_log_loss(y_valid, val_preds[:, 1])
        m.append(pipeline)
        scores.append(val_score)

In [ ]:
print('*' * 45)
print(f'Log-loss scores: {scores}')
print('*' * 45)
print(f'Log-loss scores mean: {np.mean(scores)}')

## **Submission**

In [ ]:
sample_submission.head()

In [ ]:
prediction = [0,0]
for model in m:
    prediction += model.predict_proba(test[features])

In [ ]:
# sample_submission[['class_0', 'class_1', 'class_2', 'class_3']] = prediction/len(m)
# sample_submission['class_1'] = sample_submission['class_1'] + sample_submission['class_2'] + sample_submission['class_3']
# sample_submission = sample_submission.drop(['class_2', 'class_3'], axis=1)

In [ ]:
sample_submission[['class_0', 'class_1']] = prediction/len(m)

In [ ]:
sample_submission.head()

In [ ]:
sample_submission.to_csv('submission.csv', index=False)